# Validate Node-Order Equivariance

This notebook validates that permuting node iteration order consistently transforms embeddings, targets, adjacency, graph conditioning, and the untrained transformer encoder without changing graph-level meaning.

This notebook checks whether a graph and permuted-node copies remain internally consistent through the NodeField data path. It verifies node embeddings, row-wise targets, adjacency, graph conditioning, and the untrained transformer encoder all transform equivariantly under node-row permutations.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import random
import networkx as nx
import numpy as np
import pandas as pd
import torch

from conditional_node_field_graph_generator.notebooks import configure_notebook
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from conditional_node_field_graph_generator.extensions.demo.pipeline import build_graph_generator
from conditional_node_field_graph_generator.extensions.demo.visualization import plot_networkx_graphs
from conditional_node_field_graph_generator.extensions.demo.artificial import generate_artificial_dataset


In [ ]:
RANDOM_SEED = 7
N_PERMUTATIONS = 8
EMBEDDING_DIM = 32

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

NODE_LABEL_COLORS = {
    0: '#fee2e2',
    1: '#fca5a5',
    2: '#dc2626',
    3: '#dbeafe',
    4: '#93c5fd',
    5: '#2563eb',
    6: '#dcfce7',
    7: '#86efac',
    8: '#16a34a',
}

PLOT_KWARGS = {
    'node_label_colors': NODE_LABEL_COLORS,
    'size': 3,
    'show_label': True,
    'node_size': 300,
    'node_linewidths': 2,
    'edge_width': 2,
}


In [ ]:
base_graphs, _plot_artificial_graphs = generate_artificial_dataset(
    num_graphs=1,
    cycle_length=6,
    num_cycles=1,
    path_length=4,
    num_rays=0,
    ray_length=0,
    node_alphabet_size=3,
    edge_alphabet_size=1,
    node_alphabet_kind='int',
    edge_alphabet_kind='int',
    component_specific_alphabets=True,
    seed=RANDOM_SEED,
    save_config=False,
)
base_graph = base_graphs[0]

def permute_graph_rows(graph, row_to_original_index):
    original_nodes = list(graph.nodes())
    row_to_original_node = [original_nodes[int(idx)] for idx in row_to_original_index]
    original_to_new_node = {
        original_node: new_idx
        for new_idx, original_node in enumerate(row_to_original_node)
    }
    permuted = graph.__class__()
    permuted.graph.update(dict(graph.graph))
    for new_idx, original_node in enumerate(row_to_original_node):
        permuted.add_node(new_idx, **dict(graph.nodes[original_node]))
    for u, v, attrs in graph.edges(data=True):
        permuted.add_edge(
            original_to_new_node[u],
            original_to_new_node[v],
            **dict(attrs),
        )
    return permuted

rng = np.random.default_rng(RANDOM_SEED)
row_maps = [np.arange(base_graph.number_of_nodes())]
for _ in range(N_PERMUTATIONS):
    row_maps.append(rng.permutation(base_graph.number_of_nodes()))

graphs = [permute_graph_rows(base_graph, row_map) for row_map in row_maps]

plot_networkx_graphs(
    graphs[:4],
    n_cols=4,
    titles=['identity', 'perm 1', 'perm 2', 'perm 3'],
    **PLOT_KWARGS,
)


In [ ]:
graph_generator = build_graph_generator(
    latent_embedding_dimension=EMBEDDING_DIM,
    number_of_transformer_layers=2,
    transformer_attention_head_count=4,
    maximum_epochs=1,
    batch_size=4,
    verbose=1,
    decoder_n_jobs=1,
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name='node-order-equivariance-smoke-test',
    model_dir=SAVED_GENERATOR_ROOT,
)

graph_generator.graph_vectorizer.fit(graphs)
graph_generator.node_graph_vectorizer.fit(graphs)
node_embeddings_list, graph_conditioning = graph_generator.encode(graphs)

node_label_targets = graph_generator.graphs_to_node_label_targets(graphs)
edge_label_targets, edge_label_pairs = graph_generator.graphs_to_edge_label_targets(graphs)
node_batch = graph_generator._build_node_batch(
    graphs,
    node_embeddings_list,
    node_label_targets=node_label_targets,
    edge_label_pairs=edge_label_pairs,
    edge_label_targets=edge_label_targets,
)


In [ ]:
def adjacency_in_node_iteration_order(graph):
    nodes = list(graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(nodes)}
    adj = np.zeros((len(nodes), len(nodes)), dtype=np.int64)
    for u, v in graph.edges():
        i = node_to_index[u]
        j = node_to_index[v]
        adj[i, j] = 1
        if not graph.is_directed():
            adj[j, i] = 1
    return adj

base_embeddings = np.asarray(node_embeddings_list[0], dtype=float)
base_degrees = node_batch.node_degree_targets[0, :base_graph.number_of_nodes()]
base_labels = np.asarray(node_label_targets[0], dtype=object)
base_adj = adjacency_in_node_iteration_order(graphs[0])
base_graph_embedding = np.asarray(graph_conditioning.graph_embeddings[0], dtype=float)

rows = []
for graph_idx, row_to_base in enumerate(row_maps):
    n_nodes = graphs[graph_idx].number_of_nodes()
    emb = np.asarray(node_embeddings_list[graph_idx], dtype=float)
    degrees = node_batch.node_degree_targets[graph_idx, :n_nodes]
    labels = np.asarray(node_label_targets[graph_idx], dtype=object)
    adj = adjacency_in_node_iteration_order(graphs[graph_idx])
    graph_embedding = np.asarray(graph_conditioning.graph_embeddings[graph_idx], dtype=float)

    rows.append({
        'graph_idx': graph_idx,
        'max_node_embedding_diff': float(np.max(np.abs(emb - base_embeddings[row_to_base]))),
        'degree_targets_match': bool(np.array_equal(degrees, base_degrees[row_to_base])),
        'node_labels_match': bool(np.array_equal(labels, base_labels[row_to_base])),
        'adjacency_matches': bool(np.array_equal(adj, base_adj[np.ix_(row_to_base, row_to_base)])),
        'graph_embedding_diff': float(np.max(np.abs(graph_embedding - base_graph_embedding))),
        'node_count': int(graph_conditioning.node_counts[graph_idx]),
        'edge_count': int(graph_conditioning.edge_counts[graph_idx]),
    })

consistency = pd.DataFrame(rows)
display(consistency)

assert consistency['degree_targets_match'].all()
assert consistency['node_labels_match'].all()
assert consistency['adjacency_matches'].all()
assert float(consistency['max_node_embedding_diff'].max()) < 1e-6
assert float(consistency['graph_embedding_diff'].max()) < 1e-6
print('Data path is permutation-consistent for this graph and vectorizer setup.')


In [ ]:
node_model = graph_generator.conditional_node_generator_model
node_model.setup(
    node_batch=node_batch,
    graph_conditioning=graph_conditioning,
    targets=None,
)
payload = node_model._build_processed_training_payload(node_batch, graph_conditioning)

model = node_model.model.eval()
device = next(model.parameters()).device
x = torch.tensor(payload['X_scaled'], dtype=torch.float32, device=device)
y = torch.tensor(payload['y_scaled'], dtype=torch.float32, device=device)
mask = torch.tensor(payload['mask_array'], dtype=torch.bool, device=device)

with torch.no_grad():
    latent = model._encode_with_condition(x, y, node_mask=mask).detach().cpu().numpy()

base_latent = latent[0, :base_graph.number_of_nodes()]
latent_rows = []
for graph_idx, row_to_base in enumerate(row_maps):
    n_nodes = graphs[graph_idx].number_of_nodes()
    latent_rows.append({
        'graph_idx': graph_idx,
        'max_latent_diff': float(np.max(np.abs(latent[graph_idx, :n_nodes] - base_latent[row_to_base]))),
    })

latent_consistency = pd.DataFrame(latent_rows)
display(latent_consistency)

assert float(latent_consistency['max_latent_diff'].max()) < 1e-5
print('Untrained NodeField transformer encoder is permutation-equivariant for this batch.')


If either assertion block fails, inspect which column fails first:

- `max_node_embedding_diff`: the node vectorizer rows are not aligned with `graph.nodes()` order, or features depend on node IDs.
- `degree_targets_match`, `node_labels_match`, `adjacency_matches`: the notebook's permutation construction or target extraction is inconsistent.
- `graph_embedding_diff`: graph-level vectorization is not invariant to node relabeling.
- `max_latent_diff`: the neural encoder path is not behaving equivariantly for consistently permuted node rows.